In [10]:
import cogent3
import pandas as pd
from pathlib import Path
import json

thesis_root = Path("./thesis_0.1")
species_paths = {
    "Chimpanzee": thesis_root / 'human_chimp',
    "Gorilla": thesis_root / 'human_gorilla',
    "Macaque": thesis_root / 'human_macaque',
}
primates100_path = Path("/home/richard/source/madb_data/primates100")

original_fasta_files = list(primates100_path.glob("*.fa"))

summary = []

for species, base_path in species_paths.items():
    fasta_path = base_path / "ensembl_alignments"
    not_completed_path = fasta_path / "not_completed"
    
    # original_fasta_files = list(fasta_path.glob("*.fa"))
    failed_files = list(not_completed_path.glob("*.json"))
    
    too_long = 0
    missing_species = 0
    
    for json_file in failed_files:
        try:
            with open(json_file) as f:
                record = json.load(f)
                reason = record.get("not_completed_construction", {}).get("args", [""])[2]
                if "Filtered for length" in reason:
                    too_long += 1
                elif "not in" in reason:
                    missing_species += 1
        except Exception as e:
            print(f"Failed to parse {json_file}: {e}")
    
    total = len(original_fasta_files)
    failures = too_long + missing_species
    successful = total - failures
    
    summary.append({
        "Species": species,
        "Original FASTAs": total,
        "Too Long": too_long,
        "Missing Species": missing_species,
        "Successfully Aligned": successful
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_markdown(index=False))


| Species    |   Original FASTAs |   Too Long |   Missing Species |   Successfully Aligned |
|:-----------|------------------:|-----------:|------------------:|-----------------------:|
| Chimpanzee |               121 |         46 |                10 |                     65 |
| Gorilla    |               121 |         46 |                10 |                     65 |
| Macaque    |               121 |         46 |                10 |                     65 |


In [9]:
import pathlib
import json
import cogent3
import pandas
import thesis_rec  # user-defined dataclass module

# Set paths
thesis_root = pathlib.Path("./thesis_0.1")
species_paths = {
    "Chimpanzee": thesis_root / 'human_chimp',
    "Gorilla": thesis_root / 'human_gorilla',
    "Macaque": thesis_root / 'human_macaque',
}

rows = []

for species, path in species_paths.items():
    in_dstore = cogent3.open_data_store(path / 'scored', suffix="json")
    for entry in in_dstore:
        try:
            raw = entry.read()
            data = json.loads(raw) if isinstance(raw, str) else raw
            nested_data = json.loads(data["data"])
            rec = thesis_rec.thesis_rec.from_rich_dict(nested_data)

            # Get sequence length for human from cogent3_alignment
            human_seq_len = len(rec.cogent3_alignment.get("homo_sapiens", ""))

            madb_kmer_keys = set()
            for attr in [
                rec.madb_time,
                rec.madb_score,
                rec.madb_distance,
                rec.madb_bubbles,
                rec.madb_braids,
                rec.madb_cycles,
                rec.madb_longest_braid_length,
            ]:
                madb_kmer_keys.update(int(k) for k in attr.keys())

            for k in sorted(madb_kmer_keys):
                str_k = str(k)

                if all(
                    rec.__getattribute__(attr).get(str_k) is None
                    for attr in [
                        "madb_time",
                        "madb_score",
                        "madb_distance",
                        "madb_bubbles",
                        "madb_braids",
                        "madb_cycles",
                        "madb_longest_braid_length",
                    ]
                ):
                    continue

                row = {
                    "species": species,
                    "unique_id": rec.unique_id,
                    "kmer_size": k,
                    "seq_length": human_seq_len,
                    "madb_time": rec.madb_time.get(str_k),
                    "madb_score": rec.madb_score.get(str_k),
                    "madb_distance": rec.madb_distance.get(str_k),
                    "madb_bubbles": rec.madb_bubbles.get(str_k),
                    "madb_braids": rec.madb_braids.get(str_k),
                    "madb_cycles": rec.madb_cycles.get(str_k),
                    "madb_longest_braid_length": rec.madb_longest_braid_length.get(str_k),
                    "jaccard_distance": rec.jaccard_distance.get(str_k),
                    "ensembl_pd": rec.ensembl_pd,
                    "ensembl_score": rec.ensemble_score,
                    "cogent3_time": rec.cogent3_time,
                    "cogent3_score": rec.cogent3_score,
                    "cogent3_pd": rec.cogent3_pd,
                    "madb_ungapped_sw_length": rec.madb_ungapped_smith_waterman_length,
                    "ungapped_sw_time": rec.ungapped_smith_waterman_time,
                }

                rows.append(row)

        except Exception as err:
            print(f"Failed to parse {entry}: {err}")

# Convert to DataFrame and display
df = pandas.DataFrame(rows)


# Save to disk
output_dir = thesis_root / 'df'
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / "all_alignment_data.csv"
df.to_csv(csv_path, index=False)
print(f"Saved to {csv_path}")

df


Saved to thesis_0.1/df/all_alignment_data.csv


,species,unique_id,kmer_size,seq_length,madb_time,madb_score,madb_distance,madb_bubbles,madb_braids,madb_cycles,madb_longest_braid_length,jaccard_distance,ensembl_pd,ensembl_score,cogent3_time,cogent3_score,cogent3_pd,madb_ungapped_sw_length,ungapped_sw_time
0,Chimpanzee,ENSG00000204518.fa,10,273,0.006044,269.977805,0.010989,0,1,False,273,0.185567,0.010989,269.977805,0.008650,269.977805,0.010989,273,0.005671
1,Chimpanzee,ENSG00000204518.fa,15,273,0.004888,269.977805,0.010989,0,1,False,273,0.250000,0.010989,269.977805,0.008650,269.977805,0.010989,273,0.005671
2,Chimpanzee,ENSG00000204518.fa,20,273,0.004686,269.977805,0.010989,0,1,False,273,0.312292,0.010989,269.977805,0.008650,269.977805,0.010989,273,0.005671
3,Chimpanzee,ENSG00000204518.fa,25,273,0.004399,269.977805,0.010989,0,1,False,273,0.350993,0.010989,269.977805,0.008650,269.977805,0.010989,273,0.005671
4,Chimpanzee,ENSG00000204518.fa,30,273,0.005249,269.977805,0.010989,0,1,False,273,0.384106,0.010989,269.977805,0.008650,269.977805,0.010989,273,0.005671
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1519,Macaque,ENSG00000130762.fa,75,28543,5.522588,12401.351516,0.425606,10,8,True,1883,0.997636,0.079825,6176.890372,44.390076,12411.484625,0.070657,9786,17.579811
1520,Macaque,ENSG00000130762.fa,80,28543,5.787257,12411.484625,0.589364,8,5,True,2466,0.998527,0.079825,6176.890372,44.390076,12411.484625,0.070657,9786,17.579811
1521,Macaque,ENSG00000130762.fa,85,28543,10.497331,12411.484625,0.613905,5,3,True,2409,0.999056,0.079825,6176.890372,44.390076,12411.484625,0.070657,9786,17.579811
1522,Macaque,ENSG00000130762.fa,90,28543,14.911200,12411.484625,0.006515,3,2,True,206,0.999453,0.079825,6176.890372,44.390076,12411.484625,0.070657,9786,17.579811


In [5]:
# Count unique IDs per species from all_alignment_data.csv
all_counts = df.groupby("species")["unique_id"].nunique().reset_index()
print("Unique gene counts per species (all alignments):")
print(all_counts.to_string(index=False))


Unique gene counts per species (all alignments):
   species  unique_id
Chimpanzee         34
   Gorilla         44
   Macaque         62


In [4]:
# Load the full alignment data
all_df = pandas.read_csv(thesis_root / "df" / "all_alignment_data.csv")

# Filter for just Chimpanzee and Macaque rows
chimp_df = all_df[all_df["species"] == "Chimpanzee"].copy()
macaque_df = all_df[all_df["species"] == "Macaque"].copy()

# Create a merge key on unique_id and kmer_size to identify matched entries
chimp_df["merge_key"] = chimp_df["unique_id"].str.replace(".fa", "", regex=False) + f":k" + chimp_df["kmer_size"].astype(str)
macaque_df["merge_key"] = macaque_df["unique_id"].str.replace(".fa", "", regex=False) + f":k" + macaque_df["kmer_size"].astype(str)

# Intersect merge keys
shared_keys = set(chimp_df["merge_key"]) & set(macaque_df["merge_key"])

# Filter both dataframes for shared keys
chimp_matched = chimp_df[chimp_df["merge_key"].isin(shared_keys)].copy()
macaque_matched = macaque_df[macaque_df["merge_key"].isin(shared_keys)].copy()

# Concatenate into one long dataframe
paired_df = pandas.concat([chimp_matched, macaque_matched], ignore_index=True).drop(columns=["merge_key"])

# Save to disk
paired_csv_path = thesis_root / "df" / "chimp_macaque_paired_alignments.csv"
paired_df.to_csv(paired_csv_path, index=False)
print(f"Saved paired chimp–macaque alignments to {paired_csv_path}")

paired_df

Saved paired chimp–macaque alignments to thesis_0.1/df/chimp_macaque_paired_alignments.csv


,species,unique_id,kmer_size,seq_length,madb_time,madb_score,madb_distance,madb_bubbles,madb_braids,madb_cycles,madb_longest_braid_length,jaccard_distance,ensembl_pd,ensembl_score,cogent3_time,cogent3_score,cogent3_pd,madb_ungapped_sw_length,ungapped_sw_time
0,Chimpanzee,ENSG00000204518.fa,10,273,0.006044,269.977805,0.010989,0,1,False,273,0.185567,0.010989,269.977805,0.008650,269.977805,0.010989,273,0.005671
1,Chimpanzee,ENSG00000204518.fa,15,273,0.004888,269.977805,0.010989,0,1,False,273,0.250000,0.010989,269.977805,0.008650,269.977805,0.010989,273,0.005671
2,Chimpanzee,ENSG00000204518.fa,20,273,0.004686,269.977805,0.010989,0,1,False,273,0.312292,0.010989,269.977805,0.008650,269.977805,0.010989,273,0.005671
3,Chimpanzee,ENSG00000204518.fa,25,273,0.004399,269.977805,0.010989,0,1,False,273,0.350993,0.010989,269.977805,0.008650,269.977805,0.010989,273,0.005671
4,Chimpanzee,ENSG00000204518.fa,30,273,0.005249,269.977805,0.010989,0,1,False,273,0.384106,0.010989,269.977805,0.008650,269.977805,0.010989,273,0.005671
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
579,Macaque,ENSG00000142632-0.fa,75,4932,0.287118,3094.618676,0.000000,5,4,False,102,0.995431,0.057461,3082.782951,1.159041,3094.618676,0.051898,2008,0.578316
580,Macaque,ENSG00000142632-0.fa,80,4932,0.433970,3094.618676,0.000000,3,2,False,102,0.997283,0.057461,3082.782951,1.159041,3094.618676,0.051898,2008,0.578316
581,Macaque,ENSG00000142632-0.fa,85,4932,0.634209,3094.618676,0.000000,2,1,False,102,0.998043,0.057461,3082.782951,1.159041,3094.618676,0.051898,2008,0.578316
582,Macaque,ENSG00000142632-0.fa,90,4932,0.516678,3094.618676,0.000000,2,1,False,102,0.998586,0.057461,3082.782951,1.159041,3094.618676,0.051898,2008,0.578316


In [6]:
# Count unique IDs per species from the paired dataframe
paired_counts = paired_df.groupby("species")["unique_id"].nunique().reset_index()
print("Unique gene counts per species (paired chimp–macaque):")
print(paired_counts.to_string(index=False))


Unique gene counts per species (paired chimp–macaque):
   species  unique_id
Chimpanzee         32
   Macaque         32
